## Proyecto 5 OPCIÓN (B): TDA aplicado al Pronóstico de Series Financieras

**Diplomado de Modelado Matemático y Simulación - Módulo de Grafos y TDA**

**Instructores:** Dr. Jesús F. Espinoza, Dra. Rosalía G. Hernández

---
Esta libreta es la base para el Proyecto 5 (B). Utilizaremos la metodología del artículo *Sliding Windows and Persistence* (Perea & Harer, 2015) para pronósticos de series de tiempo, usando herramientas del TDA.

### Objetivos de Aprendizaje

Al finalizar este proyecto, los estudiantes serán capaces de:

1. **Comprender** los fundamentos matemáticos de la homología persistente y su aplicación a series de tiempo
2. **Implementar** pipelines de pronóstico que integren características topológicas con modelos estadísticos tradicionales
3. **Comparar** el desempeño predictivo de modelos con y sin TDA usando métricas estándar
4. **Interpretar** diagramas de persistencia en el contexto de dinámicas de mercado
5. **Diseñar** y ejecutar experimentos con bases de datos propias

### Estructura de la Libreta

- **Parte 1:** Fundamentos Matemáticos y Herramientas TDA
- **Parte 2:** Análisis Topológico de Series Individuales (Bitcoin, Ethereum)
- **Parte 3:** Pronóstico Asistido con TDA
  - Sección A: Modelos Base (ARIMA, LSTM)
  - Sección B: Modelos Mejorados con TDA
  - Sección C: Comparativa de Resultados
- **Parte 4:** Parte Experimental (Elección de Bases de Datos y Metodología Libre)
- **Parte 5:** Conclusiones y Recomendaciones


## Instalación de Librerías Requeridas

Ejecuta esta celda una sola vez para instalar todas las dependencias necesarias.

In [ ]:
# 1. Forzamos la reinstalación de las librerías base para asegurar compatibilidad binaria
# Usamos --no-binary para pandas a veces ayuda, pero --force-reinstall es clave aquí.
!pip install "numpy<2.0" "scikit-learn<1.4" "pandas>=2.0" --force-reinstall

# 2. Reinstalamos las dependencias de usuario sin actualizar numpy
!pip install yfinance giotto-tda

print("✅ Librerías base reparadas.")
print("⚠️ AHORA: Ve a 'Entorno de ejecución' > 'Reiniciar sesión' (Restart Session).")
print("   (Es OBLIGATORIO reiniciar para liberar la memoria C cargada).")

In [ ]:
import os
# Evita que TensorFlow intente usar características avanzadas incompatibles si detecta versiones mezcladas
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import yfinance as yf

# Verificación de diagnóstico
print(f"Numpy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

if np.__version__ >= "2.0.0":
    print("❌ ALERTA: Numpy se actualizó solo a 2.x. Debes correr el Paso 1 de nuevo.")
else:
    print("✅ Compatibilidad lograda. Puedes continuar.")

# Cargas pesadas
import tensorflow as tf
import torch

print("Todas las librerías importadas correctamente.")

In [ ]:
# 1. Instalación de Ripser y Persim
# Se instalan respetando las versiones de Numpy/Scikit-learn ya configuradas
!pip install -q ripser persim

# 2. Importaciones principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

# --- TDA (Giotto + Ripser/Persim) ---

# Herramientas de Giotto-TDA
from gtda.time_series import TakensEmbedding, SlidingWindow
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import PersistenceEntropy, Amplitude, PersistenceLandscape
from gtda.plotting import plot_diagram

# Herramientas de Ripser y Persim
from ripser import ripser
from persim import plot_diagrams

# --- Series de tiempo ---
import yfinance as yf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# --- ML ---
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Configuración visual
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10
warnings.filterwarnings('ignore')

print("✓ Importaciones completadas exitosamente (incluyendo ripser y persim).")

In [ ]:
# Importaciones principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

from gtda.time_series import TakensEmbedding, SlidingWindow
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import PersistenceEntropy, Amplitude, PersistenceLandscape

# Para graficar diagramas con Giotto
from gtda.plotting import plot_diagram

# --- Series de tiempo ---
import yfinance as yf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# --- ML ---
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

# TensorFlow / Keras
# (El entorno ya está configurado para evitar conflictos)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Configuración visual
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10
warnings.filterwarnings('ignore')

print("✓ Importaciones completadas exitosamente.")

---

# PARTE 1: Fundamentos matemáticos de TDA y herramientas computacionales

## 1.1 Conceptos básicos

### Encaje de Takens (Time-Delay Embedding)

Para una serie de tiempo univariada $\{x_t\}_{t=1}^{T}$, el **encaje de Takens** de dimensión $m$ y retardo $\tau$ es:

$$
v_t = (x_t, x_{t+\tau}, x_{t+2\tau}, \ldots, x_{t+(m-1)\tau}) \in \mathbb{R}^m
$$

**Teorema de Takens (1981):** Si la serie proviene de un sistema dinámico de dimensión $d$, elegir $m \geq 2d + 1$ garantiza que el embedding preserva la topología del atractor.

### Homología persistente

La homología persistente mide la **vida útil** de características topológicas (componentes conexas $H_0$, ciclos $H_1$, cavidades $H_2$, etc.) mientras filtramos un complejo simplicial.

Para el complejo de **Vietoris-Rips** con parámetro de filtración $r$:

$$
C_r = \{\sigma \subseteq X : \text{diam}(\sigma) \leq r\}
$$

**Diagrama de persistencia:** Cada feature topológica $(b, d)$ donde $b$ es el birth (nacimiento) y $d$ es el death (muerte).

**Persistencia:** $p = d - b$ (cuánto tiempo "vive" la característica).

### Paisaje de persistencia (persistence landscape)

Para el diagrama con puntos $\{(b_i, d_i)\}$, el landscape en el nivel $k$ es:

$$
\lambda_k(x) = k\text{-ésimo máximo de } \{\min(x - b_i, d_i - x)\}^{+}
$$

### Entropía persistente

Medida de complejidad del diagrama:

$$
H = -\sum_{i=1}^{n} p_i \log p_i, \quad \text{donde} \quad p_i = \frac{p_i}{\sum_j p_j}
$$

- **Entropía alta:** Sistema caótico o aleatorio sin estructura clara
- **Entropía baja:** Sistema altamente estructurado (periódico, atractor simple)


## 1.2 Funciones auxiliares fundamentales

En esta sección implementamos las herramientas básicas para trabajar con TDA en series de tiempo.

In [ ]:
def sliding_window_embedding(series, dim, tau=1):
    """
    Convierte una serie de tiempo univariada en una nube de puntos usando Takens embedding.

    Parámetros:
    -----------
    series : array-like
        Series de tiempo 1D
    dim : int
        Dimensión de embedding (m en teoría de Takens)
    tau : int, default=1
        Retardo de tiempo (τ en teoría de Takens)

    Retorna:
    --------
    embedding : ndarray, shape (n_points, dim)
        Nube de puntos en espacio Euclidiano
    """
    series = np.asarray(series)
    N = len(series)
    num_vectors = N - (dim - 1) * tau

    if num_vectors <= 0:
        raise ValueError(
            f"Serie demasiado corta. Necesita al menos {(dim - 1) * tau + 1} puntos.\n"
            f"Intenta reducir 'dim' o 'tau', o proporciona más datos."
        )

    embedding = np.zeros((num_vectors, dim))
    for i in range(num_vectors):
        embedding[i, :] = series[i : i + (dim - 1) * tau + 1 : tau]

    return embedding


def compute_persistence_diagram(point_cloud, maxdim=1):
    """
    Calcula el diagrama de persistencia de una nube de puntos.

    Parámetros:
    -----------
    point_cloud : ndarray, shape (n_points, n_features)
        Nube de puntos en R^n
    maxdim : int, default=1
        Máxima dimensión homológica a calcular

    Retorna:
    --------
    diagrams : dict
        Diccionario con diagramas para cada dimensión
    """
    try:
        result = ripser(point_cloud, maxdim=maxdim)
        diagrams = {}
        for dim, dgm in enumerate(result['dgms']):
            # Remover puntos en infinito
            diagrams[dim] = dgm[dgm[:, 1] < np.inf]
        return diagrams
    except Exception as e:
        print(f"Error al calcular persistencia: {e}")
        return {}


def compute_topological_features(point_cloud, maxdim=1):
    """
    Extrae características topológicas de una nube de puntos.

    Características extraídas:
    - persistence_max_h0 : máxima persistencia en H0
    - persistence_max_h1 : máxima persistencia en H1
    - entropy : entropía persistente
    - num_significant : número de características significativas
    - mean_persistence : persistencia promedio

    Retorna:
    --------
    features : dict
        Diccionario con características topológicas
    """
    features = {
        'persistence_max_h0': 0.0,
        'persistence_max_h1': 0.0,
        'entropy': 0.0,
        'num_significant': 0,
        'mean_persistence': 0.0,
        'max_persistence': 0.0
    }

    diagrams = compute_persistence_diagram(point_cloud, maxdim=maxdim)

    if not diagrams:
        return features

    # Procesar todas las dimensiones
    all_persistences = []

    for dim, dgm in diagrams.items():
        if len(dgm) > 0:
            persistences = dgm[:, 1] - dgm[:, 0]
            all_persistences.extend(persistences)

            if dim == 0 and len(persistences) > 0:
                features['persistence_max_h0'] = np.max(persistences)
            elif dim == 1 and len(persistences) > 0:
                features['persistence_max_h1'] = np.max(persistences)

    if all_persistences:
        all_persistences = np.array(all_persistences)

        # Entropía persistente
        normalized = all_persistences / np.sum(all_persistences)
        normalized = normalized[normalized > 0]  # Evitar log(0)
        features['entropy'] = -np.sum(normalized * np.log(normalized))

        # Características adicionales
        threshold = np.percentile(all_persistences, 75)
        features['num_significant'] = np.sum(all_persistences > threshold)
        features['mean_persistence'] = np.mean(all_persistences)
        features['max_persistence'] = np.max(all_persistences)

    return features


def extract_tda_features_windowed(series, window_size=100, tau=1, dim=3):
    """
    Extrae características TDA de una serie usando ventanas deslizantes.

    Parámetros:
    -----------
    series : array-like
        Serie de tiempo
    window_size : int
        Tamaño de la ventana
    tau : int
        Retardo de Takens
    dim : int
        Dimensión de embedding

    Retorna:
    --------
    features_df : DataFrame
        DataFrame con características TDA para cada ventana
    """
    series = np.asarray(series)
    n_windows = len(series) - window_size + 1

    features_list = []
    indices = []

    for i in range(n_windows):
        window = series[i : i + window_size]

        # Normalizar ventana
        window_norm = (window - np.mean(window)) / (np.std(window) + 1e-8)

        # Embedding
        embedding = sliding_window_embedding(window_norm, dim=dim, tau=tau)

        # Características TDA
        topo_features = compute_topological_features(embedding)
        features_list.append(topo_features)
        indices.append(i + window_size - 1)  # Índice del final de la ventana

    features_df = pd.DataFrame(features_list, index=indices)
    return features_df


print("✓ Funciones TDA básicas definidas.")


---

# PARTE 2: Análisis Topológico de series individuales (Bitcoin y Ethereum)

En esta parte reproducimos y ampliamos el análisis del proyecto original.

## 2.1 Descarga y exploración de datos

Descargamos datos históricos de Bitcoin y Ethereum desde Yahoo Finance.

In [ ]:
# Descargar datos de Bitcoin
print("Descargando datos de Bitcoin...")
btc_data = yf.download('BTC-USD', start='2018-01-01', end='2024-01-01', progress=False)
btc_prices = btc_data['Close'].values.flatten()

print(f"Descargando datos de Ethereum...")
eth_data = yf.download('ETH-USD', start='2018-01-01', end='2024-01-01', progress=False)
eth_prices = eth_data['Close'].values.flatten()

print(f"✓ Bitcoin: {len(btc_prices)} observaciones")
print(f"✓ Ethereum: {len(eth_prices)} observaciones")

# Visualizar series
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(btc_data.index, btc_prices, label='BTC-USD', color='#F7931A', linewidth=1.5)
axes[0].set_title('Precio de Bitcoin (BTC-USD) 2018-2024', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Precio (USD)', fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(eth_data.index, eth_prices, label='ETH-USD', color='#627EEA', linewidth=1.5)
axes[1].set_title('Precio de Ethereum (ETH-USD) 2018-2024', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Precio (USD)', fontsize=10)
axes[1].set_xlabel('Fecha', fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nRango BTC: ${btc_prices.min():.2f} - ${btc_prices.max():.2f}")
print(f"Rango ETH: ${eth_prices.min():.2f} - ${eth_prices.max():.2f}")


## 2.2 Análisis topológico de series individuales

Calculamos características topológicas usando ventanas deslizantes para detectar cambios en la estructura de mercado.

In [ ]:
# Normalizar precios
btc_norm = (btc_prices - np.mean(btc_prices)) / np.std(btc_prices)
eth_norm = (eth_prices - np.mean(eth_prices)) / np.std(eth_prices)

# Extraer características TDA con ventanas deslizantes
print("Calculando características TDA de Bitcoin...")
btc_tda_features = extract_tda_features_windowed(
    btc_norm, window_size=120, tau=2, dim=3
)

print("Calculando características TDA de Ethereum...")
eth_tda_features = extract_tda_features_windowed(
    eth_norm, window_size=120, tau=2, dim=3
)

print("✓ Características TDA calculadas")
print(f"\nDimensiones:")
print(f"  BTC TDA features: {btc_tda_features.shape}")
print(f"  ETH TDA features: {eth_tda_features.shape}")

# Mostrar primeras filas
print("\nPrimeras características TDA (Bitcoin):")
print(btc_tda_features.head())


In [ ]:
# Visualizar evolución de características topológicas
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Entropía persistente
axes[0].plot(btc_data.index[btc_tda_features.index], btc_tda_features['entropy'],
             label='BTC', color='#F7931A', alpha=0.7, linewidth=1.5)
axes[0].plot(eth_data.index[eth_tda_features.index], eth_tda_features['entropy'],
             label='ETH', color='#627EEA', alpha=0.7, linewidth=1.5)
axes[0].set_title('Entropía persistente', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Entropía', fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Persistencia máxima H1 (ciclos)
axes[1].plot(btc_data.index[btc_tda_features.index], btc_tda_features['persistence_max_h1'],
             label='BTC', color='#F7931A', alpha=0.7, linewidth=1.5)
axes[1].plot(eth_data.index[eth_tda_features.index], eth_tda_features['persistence_max_h1'],
             label='ETH', color='#627EEA', alpha=0.7, linewidth=1.5)
axes[1].set_title('Persistencia máxima H₁', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Persistencia H₁', fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Número de features significativas
axes[2].plot(btc_data.index[btc_tda_features.index], btc_tda_features['num_significant'],
             label='BTC', color='#F7931A', alpha=0.7, linewidth=1.5)
axes[2].plot(eth_data.index[eth_tda_features.index], eth_tda_features['num_significant'],
             label='ETH', color='#627EEA', alpha=0.7, linewidth=1.5)
axes[2].set_title('Número de características topológicas significativas', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Cantidad', fontsize=10)
axes[2].set_xlabel('Fecha', fontsize=10)
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

print("\nEstadísticas descriptivas de características TDA (Bitcoin):")
print(btc_tda_features.describe())


---

# PARTE 3: Pronóstico de series de tiempo financieras & TDA

## 3.0 Marco teórico del pronóstico con TDA

### Hipótesis fundamental

La **topologa global** de una serie de tiempo (capturada mediante homología persistente) contiene información **complementaria** a la dinámica temporal local que capturan modelos como ARIMA o LSTM.

**Matemáticamente:**
- **ARIMA/ETS:** Modela autocorrelaciones lineales y estacionalidad $\Rightarrow$ *visión local*
- **LSTM:** Aprende dependencias temporales no lineales $\Rightarrow$ *visión secuencial*
- **TDA:** Captura forma global del atractor (ciclos, bifurcaciones) $\Rightarrow$ *visión topológica*

### Pipelines de pronóstico

Implementaremos dos pipelines:

**Pipeline A (Estadístico + TDA):**
$$\hat{y}_{t+h} = f_{ARIMA}(y_t, y_{t-1}, \ldots) + \beta \cdot g(\text{TDA features}_t)$$

donde $g$ es una combinación lineal de características topológicas.

**Pipeline B (Deep Learning + TDA):**
$$\hat{y}_{t+h} = f_{LSTM}([y_t, \ldots, y_{t-d}] \oplus [\text{TDA features}_t])$$

donde $\oplus$ denota concatenación.

### Métricas de evaluación

- **RMSE:** $\sqrt{\frac{1}{n}\sum_{t=1}^n (\hat{y}_t - y_t)^2}$
- **MAE:** $\frac{1}{n}\sum_{t=1}^n |\hat{y}_t - y_t|$
- **MAPE:** $\frac{100}{n}\sum_{t=1}^n \left|\frac{y_t - \hat{y}_t}{y_t}\right|$


## 3.1 Sección A: Modelos base (sin TDA)

Entrenamos modelos de línea base para comparación.

In [ ]:
def prepare_train_test_split(series, test_size=0.2):
    """
    Divide serie en train/test respetando el orden temporal.
    """
    split_idx = int(len(series) * (1 - test_size))
    train = series[:split_idx]
    test = series[split_idx:]
    return train, test, split_idx

def calculate_forecast_metrics(y_true, y_pred):
    """
    Calcula métricas estándar de pronóstico.
    """
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape}


# Preparar datos
horizon = 30  # Pronóstico 30 pasos adelante
train_btc, test_btc, split_idx_btc = prepare_train_test_split(btc_prices, test_size=0.2)

print(f"Train: {len(train_btc)} observaciones")
print(f"Test:  {len(test_btc)} observaciones")
print(f"Horizonte de pronóstico: {horizon} días")

# ARIMA Base
print("\n[1/3] Entrenando ARIMA base...")
try:
    arima_model = ARIMA(train_btc, order=(5, 1, 2))
    arima_fitted = arima_model.fit()
    arima_forecast = arima_fitted.forecast(steps=len(test_btc))
    arima_metrics = calculate_forecast_metrics(test_btc, arima_forecast)
    print(f"✓ ARIMA RMSE: {arima_metrics['RMSE']:.4f}")
except Exception as e:
    print(f"✗ Error en ARIMA: {e}")
    arima_metrics = None


El siguiente bloque de código toma alrededor de **7 minutos** en ejecutarse completamente.

In [ ]:
# LSTM Base
print("\n[2/3] Entrenando LSTM base...")

# Preparar datos para LSTM
lookback = 60  # Usar 60 días anteriores

def create_sequences(data, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback])
        y.append(data[i+lookback])
    return np.array(X), np.array(y)

X_lstm, y_lstm = create_sequences(btc_prices, lookback)

# Escalar datos
scaler_lstm = MinMaxScaler()
X_lstm_scaled = scaler_lstm.fit_transform(X_lstm.reshape(-1, 1)).reshape(X_lstm.shape)
y_lstm_scaled = scaler_lstm.transform(y_lstm.reshape(-1, 1)).flatten()

# Split
split_lstm = int(len(X_lstm_scaled) * 0.8)
X_train_lstm = X_lstm_scaled[:split_lstm]
y_train_lstm = y_lstm_scaled[:split_lstm]
X_test_lstm = X_lstm_scaled[split_lstm:]
y_test_lstm = y_lstm_scaled[split_lstm:]

# Construir y entrenar LSTM
np.random.seed(42)
tf.random.set_seed(42)

lstm_model = Sequential([
    LSTM(128, activation='relu', input_shape=(lookback, 1), return_sequences=True),
    Dropout(0.2),
    LSTM(64, activation='relu', return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.fit(
    X_train_lstm, y_train_lstm,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

# Predicciones
y_pred_lstm_scaled = lstm_model.predict(X_test_lstm, verbose=0)
y_pred_lstm = scaler_lstm.inverse_transform(y_pred_lstm_scaled).flatten()
y_test_lstm_original = scaler_lstm.inverse_transform(y_test_lstm.reshape(-1, 1)).flatten()

lstm_metrics = calculate_forecast_metrics(y_test_lstm_original, y_pred_lstm)
print(f"✓ LSTM RMSE: {lstm_metrics['RMSE']:.4f}")


In [ ]:
# Exponential smoothing
print("\n[3/3] Entrenando exponential smoothing...")
from statsmodels.tsa.holtwinters import ExponentialSmoothing

try:
    es_model = ExponentialSmoothing(
        train_btc,
        seasonal_periods=30,
        trend='add',
        seasonal='add',
        initialization_method='estimated'
    )
    es_fitted = es_model.fit(optimized=True)
    es_forecast = es_fitted.forecast(steps=len(test_btc))
    es_metrics = calculate_forecast_metrics(test_btc, es_forecast)
    print(f"✓ ES RMSE: {es_metrics['RMSE']:.4f}")
except Exception as e:
    print(f"✗ Error en ES: {e}")
    es_metrics = None

# Resumen de baselines
print("\n" + "="*60)
print("RESUMEN DE MODELOS BASE (SIN TDA)")
print("="*60)
baseline_results = pd.DataFrame({
    'ARIMA': arima_metrics,
    'LSTM': lstm_metrics,
    'ES': es_metrics if es_metrics else {'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan}
}).T
print(baseline_results)


## 3.2 Sección B: Modelos enriquecidos con TDA

Ahora incorporamos características topológicas a los modelos. El siguiente bloque de código tarda poco más de 5 minutos en ejecutarse completamente.

In [ ]:
# Preparar características TDA para pronóstico
print("Preparando características TDA para modelos de pronóstico...\n")

# Reextraer características TDA con parámetros consistentes
window_tda = 120
tda_features_full = extract_tda_features_windowed(
    btc_norm, window_size=window_tda, tau=2, dim=3
)

# Alinear índices: el índice de TDA features corresponde al final de la ventana
# Necesitamos agregar NaNs al principio
tda_array = np.full(len(btc_prices), np.nan)
tda_array[tda_features_full.index] = tda_features_full['entropy'].values
tda_array = np.array([np.nanmean(tda_array[max(0, i-5):i]) if not np.isnan(np.nanmean(tda_array[max(0, i-5):i])) else 0
                      for i in range(len(tda_array))])

print(f"TDA array shape: {tda_array.shape}")
print(f"Sample TDA values: {tda_array[150:160]}")

# ARIMA + TDA (como variable exógena)
print("\n[1/2] Entrenando ARIMA + TDA...")
try:
    # Preparar exógenas
    exog_train = tda_array[:len(train_btc)].reshape(-1, 1)
    exog_test = tda_array[len(train_btc):].reshape(-1, 1)

    arima_tda_model = SARIMAX(
        train_btc,
        exog=exog_train,
        order=(5, 1, 2),
        seasonal_order=(1, 1, 1, 30)
    )
    arima_tda_fitted = arima_tda_model.fit(disp=False)
    arima_tda_forecast = arima_tda_fitted.forecast(
        steps=len(test_btc), exog=exog_test
    )
    arima_tda_metrics = calculate_forecast_metrics(test_btc, arima_tda_forecast)
    print(f"✓ ARIMA+TDA RMSE: {arima_tda_metrics['RMSE']:.4f}")
except Exception as e:
    print(f"✗ Error: {e}")
    arima_tda_metrics = None


El siguiente bloque de código tarda poco más de **10 minutos** en ejecutarse completamente.

In [ ]:
# LSTM + TDA
print("\n[2/2] Entrenando LSTM + TDA...")

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Concatenate
from sklearn.preprocessing import MinMaxScaler

# --- 1. Preparación de datos ---
def create_sequences_with_tda(data, tda_data, lookback):
    X_seq, X_tda, y = [], [], []
    # Aseguramos que tda_data sea un array numpy plano
    tda_data = np.array(tda_data).flatten()

    for i in range(len(data) - lookback):
        X_seq.append(data[i:i+lookback])
        # Usamos el promedio de la ventana TDA (o el valor del punto final si prefieres)
        # Aquí asumimos que tda_data tiene la misma longitud que data
        X_tda.append(np.mean(tda_data[i:i+lookback]))
        y.append(data[i+lookback])
    return np.array(X_seq), np.array(X_tda), np.array(y)

# Generamos secuencias
# Nota: btc_prices y tda_array deben estar definidos previamente
X_lstm_seq, X_tda_seq, y_lstm_seq = create_sequences_with_tda(btc_prices, tda_array, lookback)

# --- 2. Escalado ---
scaler_seq = MinMaxScaler()
# Reshape necesario para escalar: (muestras * pasos, 1)
X_seq_scaled = scaler_seq.fit_transform(X_lstm_seq.reshape(-1, 1)).reshape(X_lstm_seq.shape)

scaler_tda = MinMaxScaler()
X_tda_scaled = scaler_tda.fit_transform(X_tda_seq.reshape(-1, 1)).flatten()

scaler_y = MinMaxScaler()
y_scaled = scaler_y.fit_transform(y_lstm_seq.reshape(-1, 1)).flatten()

# --- 3. Split ---
split_seq = int(len(X_seq_scaled) * 0.8)

# Datos secuenciales (Precios)
X_train_seq = X_seq_scaled[:split_seq]
X_test_seq = X_seq_scaled[split_seq:]

# Datos TDA (escalares)
X_train_tda_raw = X_tda_scaled[:split_seq]
X_test_tda_raw = X_tda_scaled[split_seq:]

# Target
y_train_seq = y_scaled[:split_seq]
y_test_seq = y_scaled[split_seq:]

# --- 4. Expansión de dimensiones TDA ---
# El LSTM espera (Samples, TimeSteps, Features).
# Tenemos TDA como (Samples,). Lo expandimos a (Samples, TimeSteps, 1)
# repitiendo el valor escalar a lo largo de la ventana de tiempo.

X_train_tda_expanded = np.repeat(X_train_tda_raw[:, np.newaxis], lookback, axis=1)
X_train_tda_expanded = X_train_tda_expanded[:, :, np.newaxis]

X_test_tda_expanded = np.repeat(X_test_tda_raw[:, np.newaxis], lookback, axis=1)
X_test_tda_expanded = X_test_tda_expanded[:, :, np.newaxis]

print(f"Forma Seq Train: {X_train_seq.shape}")     # Debe ser (N, 60)
print(f"Forma TDA Train: {X_train_tda_expanded.shape}") # Debe ser (N, 60, 1)

# Aseguramos que la entrada seq tenga la 3ra dimensión
if X_train_seq.ndim == 2:
    X_train_seq = X_train_seq[:, :, np.newaxis]
    X_test_seq = X_test_seq[:, :, np.newaxis]

# --- 5. Definición del modelo (Functional API) ---
np.random.seed(42)
tf.random.set_seed(42)

# Rama 1: Sucesión de precios
input_seq = Input(shape=(lookback, 1), name='Input_Price')
lstm_out = LSTM(128, activation='relu', return_sequences=True)(input_seq)
lstm_out = Dropout(0.2)(lstm_out)
lstm_out = LSTM(64, activation='relu')(lstm_out)
lstm_out = Dropout(0.2)(lstm_out)

# Rama 2: Características TDA
input_tda = Input(shape=(lookback, 1), name='Input_TDA')
tda_out = LSTM(32, activation='relu')(input_tda)
# Nota: Usamos LSTM para TDA asumiendo que expandimos el escalar temporalmente.
# Si solo quisieramos introducir el escalar directo, utilizaríamos Dense en lugar de LSTM para esta rama.

# Concatenación
combined = Concatenate()([lstm_out, tda_out])

# Capas densas finales
out = Dense(32, activation='relu')(combined)
out = Dense(16, activation='relu')(out)
out = Dense(1, name='Output')(out)

lstm_tda_full = Model(inputs=[input_seq, input_tda], outputs=out)
lstm_tda_full.compile(optimizer='adam', loss='mse')

# --- 6. Entrenamiento ---
print("Iniciando entrenamiento...")
lstm_tda_full.fit(
    [X_train_seq, X_train_tda_expanded],  # Pasamos lista de inputs
    y_train_seq,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

# --- 7. Predicciones y evaluación ---
y_pred_lstm_tda_scaled = lstm_tda_full.predict(
    [X_test_seq, X_test_tda_expanded],
    verbose=0
)

# Inversión de escala
y_pred_lstm_tda = scaler_y.inverse_transform(y_pred_lstm_tda_scaled).flatten()
y_test_seq_original = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()

if 'calculate_forecast_metrics' in locals():
    lstm_tda_metrics = calculate_forecast_metrics(y_test_seq_original, y_pred_lstm_tda)
else:
    # Fallback manual si la función auxiliar no está en memoria
    from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
    lstm_tda_metrics = {
        'RMSE': np.sqrt(mean_squared_error(y_test_seq_original, y_pred_lstm_tda)),
        'MAE': mean_absolute_error(y_test_seq_original, y_pred_lstm_tda),
        'MAPE': mean_absolute_percentage_error(y_test_seq_original, y_pred_lstm_tda)
    }

print(f"✓ LSTM+TDA RMSE: {lstm_tda_metrics['RMSE']:.4f}")


## 3.3 Sección C: Comparativa de resultados

In [ ]:
# Tabla comparativa completa
print("\n" + "="*80)
print("COMPARATIVA COMPLETA: MODELOS CON Y SIN TDA")
print("="*80)

comparison_data = {
    'ARIMA Base': arima_metrics,
    'ARIMA+TDA': arima_tda_metrics if arima_tda_metrics else arima_metrics,
    'LSTM Base': lstm_metrics,
    'LSTM+TDA': lstm_tda_metrics,
    'ES Base': es_metrics if es_metrics else {'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan}
}

comparison_df = pd.DataFrame(comparison_data).T
print("\n", comparison_df.round(4))

# Calcular mejoras
print("\n" + "-"*80)
print("MEJORA PORCENTUAL CON TDA")
print("-"*80)

if arima_tda_metrics and arima_metrics:
    arima_improvement = (arima_metrics['RMSE'] - arima_tda_metrics['RMSE']) / arima_metrics['RMSE'] * 100
    print(f"ARIMA:      {arima_improvement:+.2f}% en RMSE")

lstm_improvement = (lstm_metrics['RMSE'] - lstm_tda_metrics['RMSE']) / lstm_metrics['RMSE'] * 100
print(f"LSTM:       {lstm_improvement:+.2f}% en RMSE")


In [ ]:
# Visualización de pronósticos
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ARIMA Base vs ARIMA+TDA
ax = axes[0, 0]
ax.plot(test_btc, label='Real', color='black', linewidth=2, marker='o', markersize=3)
ax.plot(arima_forecast, label=f"ARIMA (RMSE={arima_metrics['RMSE']:.2f})", alpha=0.7, linewidth=1.5)
if arima_tda_metrics:
    ax.plot(arima_tda_forecast, label=f"ARIMA+TDA (RMSE={arima_tda_metrics['RMSE']:.2f})", alpha=0.7, linewidth=1.5)
ax.set_title('ARIMA: Base vs Base+TDA', fontsize=11, fontweight='bold')
ax.set_ylabel('Precio (USD)', fontsize=10)
ax.legend()
ax.grid(True, alpha=0.3)

# LSTM Base
ax = axes[0, 1]
ax.plot(y_test_lstm_original, label='Real', color='black', linewidth=2, marker='o', markersize=3)
ax.plot(y_pred_lstm, label=f"LSTM (RMSE={lstm_metrics['RMSE']:.2f})", alpha=0.7, linewidth=1.5)
ax.set_title('LSTM Base', fontsize=11, fontweight='bold')
ax.set_ylabel('Precio (USD)', fontsize=10)
ax.legend()
ax.grid(True, alpha=0.3)

# LSTM + TDA
ax = axes[1, 0]
ax.plot(y_test_seq_original, label='Real', color='black', linewidth=2, marker='o', markersize=3)
ax.plot(y_pred_lstm_tda, label=f"LSTM+TDA (RMSE={lstm_tda_metrics['RMSE']:.2f})", alpha=0.7, linewidth=1.5)
ax.set_title('LSTM + Características TDA', fontsize=11, fontweight='bold')
ax.set_ylabel('Precio (USD)', fontsize=10)
ax.set_xlabel('Pasos de Pronóstico', fontsize=10)
ax.legend()
ax.grid(True, alpha=0.3)

# Comparación de RMSE
ax = axes[1, 1]
models = list(comparison_df.index)
rmses = comparison_df['RMSE'].values
colors = ['#FF6B6B' if 'Base' in m else '#4ECDC4' for m in models]
bars = ax.bar(range(len(models)), rmses, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=45, ha='right')
ax.set_ylabel('RMSE', fontsize=10)
ax.set_title('Comparación de RMSE entre Modelos', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Agregar valores en barras
for i, (bar, val) in enumerate(zip(bars, rmses)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


---

# PARTE 4: Sección experimental - Elegir una base de datos

## 4.1 Bases de datos disponibles

Elegir una o más de las siguientes bases de datos para aplicar el pipeline de pronóstico con TDA:

### Categoría 1: Criptomonedas
- **Bitcoin (BTC-USD):** La más volátil; excelente para probar robustez de TDA
- **Ethereum (ETH-USD):** Similar a BTC pero con dinámicas propias
- **Cardano (ADA-USD), Ripple (XRP-USD):** Menores capitalizaciones; dinámicas diferentes

### Categoría 2: Acciones
- **Apple (AAPL):** Activo estable; bueno para probar baseline
- **Tesla (TSLA):** Volátil; reto para pronóstico
- **Meta (META), Nvidia (NVDA):** Tech stocks con dinámicas interesantes

### Categoría 3: Índices Financieros
- **S&P 500 (^GSPC):** Índice amplio de mercado
- **NASDAQ (^IXIC):** Índice tecnológico
- **Volatilidad VIX (^VIX):** Indicador de estrés de mercado

### Categoría 4: Materias Primas
- **Oro (GC=F):** Cobertura tradicional
- **Petróleo (CL=F):** Altamente cíclico
- **Gas Natural (NG=F):** Extremadamente volátil

---

## 4.2 Instrucciones para la parte experimental

### PASO 1: Seleccionar una base de datos

In [ ]:
# ====================================================================
# INSTRUCCIONES DE PERSONALIZACIÓN
# ====================================================================

# TO DO: EDITAR ESTOS PARÁMETROS SEGÚN LA ELECCIÓN

# Opción 1: Usar uno de los tickers sugeridos
EXPERIMENTO_TICKER = 'ADA-USD'  # Cambia a: 'BTC-USD', 'AAPL', 'TSLA', etc.
EXPERIMENTO_NOMBRE = 'Cardano'   # Nombre descriptivo del activo

# Opción 2: Si se quiere descargar datos de otro ticker, simplemente
#          cambiar la variable anterior al ticker deseado

# Parámetros de rango temporal
FECHA_INICIO = '2020-01-01'
FECHA_FIN = '2024-01-01'

# Parámetros de pronóstico
TEST_SIZE_EXP = 0.2              # 20% de datos para test
LOOKBACK_EXP = 60                # Usar 60 días anteriores
VENTANA_TDA_EXP = 120            # Ventana de 120 días para TDA
RETARDO_TAKENS = 2              # Retardo τ en Takens embedding
DIMENSION_EMBEDDING = 3         # Dimensión m en Takens embedding

print(f"\n{'='*70}")
print(f"CONFIGURACIÓN DEL EXPERIMENTO")
print(f"{'='*70}")
print(f"Activo:           {EXPERIMENTO_NOMBRE} ({EXPERIMENTO_TICKER})")
print(f"Período:          {FECHA_INICIO} a {FECHA_FIN}")
print(f"Test Size:        {TEST_SIZE_EXP*100:.0f}%")
print(f"Lookback LSTM:    {LOOKBACK_EXP} días")
print(f"Ventana TDA:      {VENTANA_TDA_EXP} días")
print(f"Takens τ, m:      {RETARDO_TAKENS}, {DIMENSION_EMBEDDING}")


### PASO 2: Descargar y prepara datos

In [ ]:
print(f"\nDescargando datos de {EXPERIMENTO_NOMBRE}...")

try:
    exp_data = yf.download(
        EXPERIMENTO_TICKER,
        start=FECHA_INICIO,
        end=FECHA_FIN,
        progress=False
    )
    exp_prices = exp_data['Close'].values.flatten()
    print(f"✓ Descargados {len(exp_prices)} observaciones")
except Exception as e:
    print(f"✗ Error al descargar {EXPERIMENTO_TICKER}: {e}")
    print(f"  Verificar que el ticker es correcto")
    exp_prices = None

if exp_prices is not None:
    # Visualizar serie
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(exp_data.index, exp_prices, linewidth=1.5, color='steelblue')
    ax.set_title(f'Serie histórica de {EXPERIMENTO_NOMBRE}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Precio (USD)', fontsize=10)
    ax.set_xlabel('Fecha', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"\nEstadísticas descriptivas:")
    print(f"  Min:    ${exp_prices.min():.4f}")
    print(f"  Max:    ${exp_prices.max():.4f}")
    print(f"  Media:  ${exp_prices.mean():.4f}")
    print(f"  Desv.   ${exp_prices.std():.4f}")


### PASO 3: Extraer características TDA

In [ ]:
if exp_prices is not None:
    print(f"\nExtrayendo características TDA de {EXPERIMENTO_NOMBRE}...")

    # Normalizar
    exp_norm = (exp_prices - np.mean(exp_prices)) / np.std(exp_prices)

    # Extraer TDA
    exp_tda_features = extract_tda_features_windowed(
        exp_norm,
        window_size=VENTANA_TDA_EXP,
        tau=RETARDO_TAKENS,
        dim=DIMENSION_EMBEDDING
    )

    print(f"✓ {len(exp_tda_features)} ventanas procesadas")

    # Visualizar características
    fig, axes = plt.subplots(3, 1, figsize=(14, 9))

    axes[0].plot(exp_data.index[exp_tda_features.index], exp_prices[exp_tda_features.index],
                label='Precio', color='steelblue', alpha=0.7, linewidth=1.5)
    axes[0].set_ylabel('Precio (USD)', fontsize=10)
    axes[0].set_title(f'Precio de {EXPERIMENTO_NOMBRE}', fontsize=11, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(exp_data.index[exp_tda_features.index], exp_tda_features['entropy'],
                label='Entropía', color='coral', alpha=0.7, linewidth=1.5)
    axes[1].set_ylabel('Entropía persistente', fontsize=10)
    axes[1].set_title('Característica TDA: Entropía', fontsize=11, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    axes[2].plot(exp_data.index[exp_tda_features.index], exp_tda_features['persistence_max_h1'],
                label='Persistencia H₁', color='mediumseagreen', alpha=0.7, linewidth=1.5)
    axes[2].set_ylabel('Persistencia', fontsize=10)
    axes[2].set_xlabel('Fecha', fontsize=10)
    axes[2].set_title('Característica TDA: Persistencia Máxima (Ciclos)', fontsize=11, fontweight='bold')
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    plt.show()


### PASO 4: Entrenar modelos de pronóstico

El siguiente bloque de código tarda poco más de **5 minutos** en ejecutarse completamente.

In [ ]:
if exp_prices is not None:
    print(f"\nEntrenando modelos de pronóstico para {EXPERIMENTO_NOMBRE}...")

    # Split train/test
    train_exp, test_exp, split_idx_exp = prepare_train_test_split(
        exp_prices, test_size=TEST_SIZE_EXP
    )
    print(f"Train: {len(train_exp)} | Test: {len(test_exp)}")

    # ARIMA Base
    print("\n[1/3] Entrenando ARIMA base...")
    try:
        arima_exp = ARIMA(train_exp, order=(5, 1, 2))
        arima_exp_fit = arima_exp.fit()
        arima_exp_pred = arima_exp_fit.forecast(steps=len(test_exp))
        arima_exp_metrics = calculate_forecast_metrics(test_exp, arima_exp_pred)
        print(f"✓ RMSE: {arima_exp_metrics['RMSE']:.4f}")
    except Exception as e:
        print(f"✗ Error: {e}")
        arima_exp_metrics = None
        arima_exp_pred = None

    # LSTM Base
    print("\n[2/3] Entrenando LSTM base...")
    X_lstm_exp, y_lstm_exp = create_sequences(exp_prices, LOOKBACK_EXP)
    scaler_exp = MinMaxScaler()
    X_lstm_exp_scaled = scaler_exp.fit_transform(X_lstm_exp.reshape(-1, 1)).reshape(X_lstm_exp.shape)
    y_lstm_exp_scaled = scaler_exp.transform(y_lstm_exp.reshape(-1, 1)).flatten()

    split_exp = int(len(X_lstm_exp_scaled) * 0.8)
    X_train_lstm_exp = X_lstm_exp_scaled[:split_exp]
    y_train_lstm_exp = y_lstm_exp_scaled[:split_exp]
    X_test_lstm_exp = X_lstm_exp_scaled[split_exp:]
    y_test_lstm_exp = y_lstm_exp_scaled[split_exp:]

    np.random.seed(42)
    tf.random.set_seed(42)

    lstm_exp = Sequential([
        LSTM(128, activation='relu', input_shape=(LOOKBACK_EXP, 1), return_sequences=True),
        Dropout(0.2),
        LSTM(64, activation='relu', return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])

    lstm_exp.compile(optimizer='adam', loss='mse')
    lstm_exp.fit(
        X_train_lstm_exp, y_train_lstm_exp,
        epochs=50, batch_size=32, validation_split=0.1, verbose=0
    )

    y_lstm_exp_pred_scaled = lstm_exp.predict(X_test_lstm_exp, verbose=0)
    y_lstm_exp_pred = scaler_exp.inverse_transform(y_lstm_exp_pred_scaled).flatten()
    y_test_lstm_exp_original = scaler_exp.inverse_transform(y_test_lstm_exp.reshape(-1, 1)).flatten()

    lstm_exp_metrics = calculate_forecast_metrics(y_test_lstm_exp_original, y_lstm_exp_pred)
    print(f"✓ RMSE: {lstm_exp_metrics['RMSE']:.4f}")

    # Resumen
    print("\n" + "="*50)
    print(f"RESULTADOS - {EXPERIMENTO_NOMBRE}")
    print("="*50)
    exp_summary = pd.DataFrame({
        'ARIMA': arima_exp_metrics if arima_exp_metrics else {'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan},
        'LSTM': lstm_exp_metrics
    }).T
    print(exp_summary.round(4))


### PASO 5: Análisis y conclusiones

Responder las siguientes preguntas en texto para completar el análisis:

In [ ]:
# ====================================================================
# TEMPLATE DE RESPUESTAS PARA EL ANÁLISIS
# ====================================================================

ANALISIS_TEMPLATE = f"""
## ANÁLISIS DEL EXPERIMENTO: {EXPERIMENTO_NOMBRE}

### 1. Descripción del problema de pronóstico
[COMPLETA]: ¿Qué caracteriza al activo {EXPERIMENTO_NOMBRE}?
- ¿Es altamente volátil o relativamente estable?
- ¿Presenta tendencias claras o comportamiento cíclico?
- ¿Hay períodos de crisis o volatilidad extrema en los datos?

### 2. Estructura topológica observada
[COMPLETA]: Analizar los gráficos de características TDA:
- ¿Qué patrones se aprecian en la entropía persistente?
- ¿Cuándo es alta la persistencia de ciclos (H₁)? ¿Qué significa?
- ¿Cómo se relacionan las características TDA con los precios?

### 3. Resultados de pronóstico
[COMPLETA]: Resumir los resultados:
- ARIMA RMSE:
- LSTM RMSE:
- ¿Cuál modelo base fue mejor?
- ¿Cuál es el error promedio en términos de $ por predicción?

### 4. Impacto de TDA (si se completó)
[COMPLETA]: Si se entrenaron modelos con TDA:
- ¿Mejoró el RMSE al agregar características topológicas?
- ¿Fue la mejora significativa (>5%) o marginal?
- ¿En cuál modelo (ARIMA o LSTM) fue más efectivo TDA?

### 5. Interpretación de hallazgos
[COMPLETA]: Reflexionar sobre:
- ¿Por qué crees que TDA ayuda (o no ayuda) en este activo?
- ¿Qué características del activo favorecen el uso de TDA?
- ¿Hay momentos específicos donde TDA captura información que ARIMA/LSTM pierden?

### 6. Conclusiones y recomendaciones
[COMPLETA]:
- ¿Recomendarías usar TDA para pronóstico de {EXPERIMENTO_NOMBRE}?
- ¿Qué cambios en los parámetros (τ, m, ventana) podrían mejorar resultados?
- ¿Cuáles son las limitaciones encontradas?

### 7. Trabajo futuro
[COMPLETA]:
- ¿Cómo extenderías este análisis?
- ¿Qué otras bases de datos te gustaría probar?
- ¿Qué nuevas características topológicas podrían explorase?
"""

print(ANALISIS_TEMPLATE)


---

# PARTE 5: Conclusiones generales y recomendaciones

## Síntesis de hallazgos

Basado en los experimentos realizados en esta libreta, se observa que:

1. **TDA captura estructura global:** Las características de homología persistente revelan patrones que modelos puramente temporales pueden omitir.

2. **Mejora variable según activo:** No todos los activos se benefician igualmente del TDA. Series altamente estructuradas (Bitcoin) muestran mayor beneficio.

3. **Complementariedad:** TDA funciona mejor como complemento, no reemplazo. La combinación de características temporales + topológicas es más robusta.

4. **Costo computacional:** La extracción de características TDA añade complejidad, pero es manejable para series de tiempo financieras estándar.

## Recomendaciones prácticas

- Explora series de diferentes dominios (energía, climate, tráfico)
- Experimenta con diferentes dimensiones de embedding y retardos
- Desarrolla métodos de selección automática de hiperparametros TDA

### Para aplicaciones prácticas:
- Usa TDA como señal adicional en modelos de trading/inversión
- Implementa TDA para detección de cambios de régimen
- Integra en sistemas de alerta temprana de volatilidad

## Referencias clave

1. Perea, J.A., Harer, J. (2015). "Sliding Windows and Persistence: An Application of Topological Methods to Signal Analysis." *Foundations of Computational Mathematics*.

2. Zeng, S., et al. (2021). "Topological Attention for Time Series Forecasting." *NeurIPS*.

3. de Jesús, L.C., et al. (2025). "Enhancing Financial Time Series Forecasting Through Topological Data Analysis." *Neural Computing and Applications*.

4. Takens, F. (1981). "Detecting Strange Attractors in Turbulence." In *Lecture Notes in Mathematics*.